# Bodycam Pipeline — Run with Real Data

Two ways to feed the pipeline:

| Mode | When to use |
|------|------------|
| **A — Brave Discovery** | You have search keywords and want the pipeline to find candidates automatically |
| **B — Curated Input** | You already have specific URLs/videos you want to process |

Run the **Setup** and **API Key** cells first, then pick your mode.

In [ ]:
# --- Setup: clone repo and cd into it ---
import os

REPO = "https://github.com/jj55222/NEWS--VIEWS.git"
BRANCH = "claude/debug-colab-script-0MCwX"

if not os.path.isdir("NEWS--VIEWS"):
    !git clone -b {BRANCH} {REPO}
else:
    print("Repo already cloned.")

%cd NEWS--VIEWS
!git checkout {BRANCH} && git pull origin {BRANCH}
print("\nReady.")

In [ ]:
# --- Configure Brave API Key ---
# Option 1 (recommended): Add BRAVE_API_KEY to Colab Secrets
#   Click the key icon in the left sidebar -> Add secret named BRAVE_API_KEY
# Option 2: Paste your key directly below

import os

MANUAL_KEY = ""  # <-- paste your Brave API key here if not using Colab Secrets

try:
    from google.colab import userdata
    key = userdata.get("BRAVE_API_KEY")
    if key:
        os.environ["BRAVE_API_KEY"] = key
        print("Loaded BRAVE_API_KEY from Colab Secrets.")
    else:
        raise ValueError("empty")
except Exception:
    if MANUAL_KEY:
        os.environ["BRAVE_API_KEY"] = MANUAL_KEY
        print("Using manually provided BRAVE_API_KEY.")
    else:
        print("WARNING: No BRAVE_API_KEY set.")
        print("  -> Add it to Colab Secrets (key icon in sidebar)")
        print("  -> Or set MANUAL_KEY in this cell")
        print("  -> Get a free key at https://api.search.brave.com")

---
## Mode A: Brave Keyword Discovery

Edit the `KEYWORDS` list below with your search terms. The pipeline will:
1. Search Brave for each keyword
2. Score results for raw-footage likelihood
3. Run each candidate through normalize -> enrich -> score -> packetize

In [ ]:
# --- Mode A: Run pipeline from Brave keyword search ---

KEYWORDS = [
    "bodycam use of force 2024",
    "police body camera shooting arrest",
    "deputy bodycam pursuit",
]
RESULTS_PER_KEYWORD = 5
OUTPUT_DIR = "runs/brave_run"

import subprocess, json, shutil, os

# Clean previous run
if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

cmd = [
    "python", "-m", "src.pipeline",
    "--keywords", *KEYWORDS,
    "--count-per-keyword", str(RESULTS_PER_KEYWORD),
    "--output", OUTPUT_DIR,
]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

# Show results
metrics_path = os.path.join(OUTPUT_DIR, "batch_metrics.json")
if os.path.isfile(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("\n=== Batch Metrics ===")
    print(json.dumps(metrics, indent=2))

    packets_path = os.path.join(OUTPUT_DIR, "packets_index.json")
    if os.path.isfile(packets_path):
        with open(packets_path) as f:
            packets = json.load(f)
        print(f"\n=== {len(packets)} Packets Generated ===")
        for i, p in enumerate(packets):
            snap = p.get("score_snapshot", {})
            print(f"  [{i}] {p.get('routing_status', '?'):20s} | "
                  f"story={snap.get('story_value_score', '?'):>3} "
                  f"research={snap.get('researchability_score', '?'):>3} | "
                  f"{p.get('source_title', '')[:60]}")
else:
    print("No output produced. Check errors above.")

---
## Mode B: Curated Input

Use this when you already have specific URLs. Fill in the template below with your real data.

**Required fields:** `source_url`, `title`, `publisher`, `published_date`, `media_type`

**Optional `hints`** (improve scoring): `agency`, `incident_date`, `location`, `people`, `incident_type`, `allegations_or_charges`, `transcript_search_anchors`

In [ ]:
# --- Mode B: Run pipeline from curated input ---
# Replace the example below with your real data.
# Add more items to the list for batch processing.

import json, subprocess, shutil, os

candidates = [
    {
        "source_url": "PASTE_URL_HERE",          # YouTube link, news article, etc.
        "title": "Video or article title",
        "description": "Brief description of the content",
        "publisher": "Channel or outlet name",
        "published_date": "2024-06-15",            # YYYY-MM-DD
        "media_type": "video",                      # "video" or "web"
        "transcript_available": False,
        "raw_footage_likelihood": 0.85,             # 0.0 to 1.0
        "watermark_likelihood": 0.1,                # 0.0 to 1.0
        "hints": {
            "agency": "Department name",             # or "UNKNOWN"
            "incident_date": "2024-06-10",          # or "UNKNOWN"
            "location": "City, State",              # or "UNKNOWN"
            "people": ["Officer Name"],              # or []
            "incident_type": "use of force",        # or "UNSPECIFIED"
            "allegations_or_charges": ["excessive force"],  # or []
            "transcript_search_anchors": ["taser deployed", "stop resisting"]  # or []
        }
    },
    # Add more candidates here:
    # { "source_url": "...", "title": "...", ... },
]

OUTPUT_DIR = "runs/curated_run"
INPUT_FILE = "input_curated.json"

# Write input and run
with open(INPUT_FILE, "w") as f:
    json.dump(candidates, f, indent=2)

if os.path.isdir(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

cmd = ["python", "-m", "src.pipeline", "--input", INPUT_FILE, "--output", OUTPUT_DIR]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

# Show results
metrics_path = os.path.join(OUTPUT_DIR, "batch_metrics.json")
if os.path.isfile(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("\n=== Batch Metrics ===")
    print(json.dumps(metrics, indent=2))

    packets_path = os.path.join(OUTPUT_DIR, "packets_index.json")
    if os.path.isfile(packets_path):
        with open(packets_path) as f:
            packets = json.load(f)
        print(f"\n=== {len(packets)} Packets Generated ===")
        for i, p in enumerate(packets):
            snap = p.get("score_snapshot", {})
            print(f"  [{i}] {p.get('routing_status', '?'):20s} | "
                  f"story={snap.get('story_value_score', '?'):>3} "
                  f"research={snap.get('researchability_score', '?'):>3} | "
                  f"{p.get('source_title', '')[:60]}")
else:
    print("No output produced. Check errors above.")

In [ ]:
# --- Explore Results: view any packet in detail ---

import json, os

# Change this to match whichever run you want to inspect
RUN_DIR = "runs/brave_run"  # or "runs/curated_run"

packets_path = os.path.join(RUN_DIR, "packets_index.json")
if not os.path.isfile(packets_path):
    print(f"No packets_index.json in {RUN_DIR}. Run the pipeline first.")
else:
    with open(packets_path) as f:
        packets = json.load(f)

    if not packets:
        print("No packets in this run.")
    else:
        # Summary table
        print(f"{'#':>3}  {'Status':20s}  {'Agency':25s}  {'Location':20s}  {'Story':>5}  {'Research':>8}")
        print("-" * 90)
        for i, p in enumerate(packets):
            snap = p.get("score_snapshot", {})
            print(f"{i:>3}  {p.get('routing_status', '?'):20s}  "
                  f"{p.get('agency', '?')[:25]:25s}  "
                  f"{p.get('location', '?')[:20]:20s}  "
                  f"{snap.get('story_value_score', '?'):>5}  "
                  f"{snap.get('researchability_score', '?'):>8}")

        # Pick one to view in full
        PACKET_INDEX = 0  # <-- change this to view a different packet
        print(f"\n=== Packet [{PACKET_INDEX}] Full Detail ===")
        print(json.dumps(packets[PACKET_INDEX], indent=2))